In [1]:
!pip install -q -U transformers accelerate requests

import os
import re
import json
import glob
import torch
from collections import Counter
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head
layers = model.model.language_model.layers

n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = layers[0].self_attn.o_proj.in_features // n_head
rms_eps = model.config.text_config.rms_norm_eps
assert len(layers) == n_layer
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim,", model.config.text_config.num_key_value_heads, "kv heads")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

28 language model layers, 28 heads, 128 head dim, 4 kv heads


In [3]:
!wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
!wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
!unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
!unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
for zpath in ["/kaggle/working/VG_100K.zip", "/kaggle/working/VG_100K_2.zip"]:
    if os.path.exists(zpath):
        os.remove(zpath)
print("Visual Genome downloaded, extracted, zips removed")

--2026-10-02 10:30:31--  https://cs.stanford.edu/people/rak248/VG_100K/images.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9730308001 (9.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K.zip’

/kaggle/working/VG_ 100%[===================>]   9.06G  21.4MB/s    in 7m 49s  

2026-10-02 10:38:21 (19.8 MB/s) - ‘/kaggle/working/VG_100K.zip’ saved [9730308001/9730308001]

--2026-10-02 10:38:21--  https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip
Resolving cs.stanford.edu (cs.stanford.edu)... 171.64.64.64
Connecting to cs.stanford.edu (cs.stanford.edu)|171.64.64.64|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5471658058 (5.1G) [application/zip]
Saving to: ‘/kaggle/working/VG_100K_2.zip’

/kaggle/working/VG_ 100%[===================>]   5.10G  13.2MB/s    in 5m 12s  

2026-10-02 10:43:34 (16.7 MB

In [4]:
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None


def resolve_image_path(q: dict) -> str:
    qid = q["question_id"]
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    elif qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None

In [5]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"

with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)

assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}
print(len(all_questions), "questions loaded")

17492 questions loaded


In [6]:
def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))


def cache_files(t):
    files = sorted(
        glob.glob(f"/kaggle/input/datasets/*/question-aware-cache-qwen25vl-{t}/{t}_batch_*.pt"),
        key=batch_number,
    )
    assert files, f"attach question-aware-cache-qwen25vl-{t}"
    return files


SOURCES = [
    ("attribute/amber", "attribute", "amber_"),
    ("relation/amber", "relation", "amber_"),
    ("relation/reefknot", "relation", "reefknot_"),
]
N_PER_SOURCE = 16
YESNO = ("yes", "no")

answer_ids = {}
top_counts = {}
samples = {}
for label, t, prefix in SOURCES:
    picked = []
    top_counts[label] = Counter()
    for path in cache_files(t):
        batch = torch.load(path, mmap=True)
        for e in batch:
            if not e["question_id"].startswith(prefix):
                continue
            top_id = int(e["answer_logits"].argmax())
            top_counts[label][top_id] += 1
            answer = e["answer_text"].strip().lower()
            if answer in YESNO:
                answer_ids.setdefault(prefix, {"yes": [], "no": []})[answer].append(top_id)
            if len(picked) < N_PER_SOURCE:
                picked.append({
                    "question_id": e["question_id"],
                    "prefix": prefix,
                    "answer_text": e["answer_text"],
                    "top_id": top_id,
                    "answer_logits": e["answer_logits"].clone(),
                })
        del batch
        if len(picked) == N_PER_SOURCE:
            break
    samples[label] = picked

PAIRS = {}
for prefix, d in answer_ids.items():
    yes_id = Counter(d["yes"]).most_common(1)[0][0]
    no_id = Counter(d["no"]).most_common(1)[0][0]
    PAIRS[prefix] = (yes_id, no_id)
    yes_purity = d["yes"].count(yes_id) / len(d["yes"])
    no_purity = d["no"].count(no_id) / len(d["no"])
    print(f"{prefix:10s} yes_id={yes_id} {processor.tokenizer.decode(yes_id)!r} purity={yes_purity:.4f} | "
          f"no_id={no_id} {processor.tokenizer.decode(no_id)!r} purity={no_purity:.4f}")
    assert yes_purity > 0.9 and no_purity > 0.9, f"mixed casing for {prefix}"
assert set(PAIRS) == {"amber_", "reefknot_"}, set(PAIRS)

for label, _, prefix in SOURCES:
    total = sum(top_counts[label].values())
    in_pair = sum(top_counts[label][i] for i in PAIRS[prefix])
    print(f"{label:18s} top-1 in the yes/no pair: {in_pair}/{total} ({in_pair / total:.4f}) over the scanned batches")

assert all(len(v) == N_PER_SOURCE for v in samples.values()), {k: len(v) for k, v in samples.items()}
for entries in samples.values():
    for e in entries:
        image_path = resolve_image_path(question_by_id[e["question_id"]])
        assert image_path is not None and os.path.exists(image_path), f"missing image for {e['question_id']}"
print({k: len(v) for k, v in samples.items()})

amber_     yes_id=9454 'Yes' purity=1.0000 | no_id=2753 'No' purity=1.0000
reefknot_  yes_id=9693 'yes' purity=0.9796 | no_id=2152 'no' purity=0.9730
attribute/amber    top-1 in the yes/no pair: 202/250 (0.8080) over the scanned batches
relation/amber     top-1 in the yes/no pair: 234/250 (0.9360) over the scanned batches
relation/reefknot  top-1 in the yes/no pair: 84/86 (0.9767) over the scanned batches
{'attribute/amber': 16, 'relation/amber': 16, 'relation/reefknot': 16}


In [7]:
_state = {"prefix": None}
_u_cache = {}

U = {
    prefix: ln_f.weight.float().cpu() * (lm_head.weight[yes_id] - lm_head.weight[no_id]).float().cpu()
    for prefix, (yes_id, no_id) in PAIRS.items()
}


def u_on(device):
    key = (_state["prefix"], device)
    if key not in _u_cache:
        _u_cache[key] = U[_state["prefix"]].to(device)
    return _u_cache[key]


HEAD_PROJ = {}
with torch.no_grad():
    for prefix in PAIRS:
        _state["prefix"] = prefix
        HEAD_PROJ[prefix] = []
        for layer in layers:
            w = layer.self_attn.o_proj.weight
            HEAD_PROJ[prefix].append(
                w.float().T.contiguous().view(n_head, head_dim, hidden_dim) @ u_on(w.device)
            )

_dla_cache = {}


def _embed_hook(module, args, kwargs):
    h = args[0] if args else kwargs["hidden_states"]
    _dla_cache["embed"] = h[0, -1].float() @ u_on(h.device)


def _make_attn_hook(l):
    def hook(module, args):
        z = args[0][0, -1].float().reshape(n_head, head_dim)
        _dla_cache[f"attn_{l}"] = (z * HEAD_PROJ[_state["prefix"]][l]).sum(-1)
    return hook


def _make_mlp_hook(l):
    def hook(module, args, output):
        _dla_cache[f"mlp_{l}"] = output[0, -1].float() @ u_on(output.device)
    return hook


def _final_norm_hook(module, args):
    h = args[0][0, -1].float()
    _dla_cache["resid_dot"] = h @ u_on(h.device)
    _dla_cache["rms"] = torch.sqrt(h.pow(2).mean() + rms_eps)


_hook_handles = [layers[0].register_forward_pre_hook(_embed_hook, with_kwargs=True)]
for l, layer in enumerate(layers):
    _hook_handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_make_attn_hook(l)))
    _hook_handles.append(layer.mlp.register_forward_hook(_make_mlp_hook(l)))
_hook_handles.append(ln_f.register_forward_pre_hook(_final_norm_hook))
print(len(_hook_handles), "hooks registered")

58 hooks registered


In [8]:
@torch.no_grad()
def dla_question(image_path, question_text, prefix):
    _state["prefix"] = prefix
    yes_id, no_id = PAIRS[prefix]
    image = load_image(image_path)
    inputs = build_inputs(image, text=question_text)

    _dla_cache.clear()
    outputs = model(**inputs)
    last_logits = outputs.logits[0, -1].float().cpu()

    return {
        "embed": _dla_cache["embed"].cpu(),
        "attn": torch.stack([_dla_cache[f"attn_{l}"].cpu() for l in range(n_layer)]),
        "mlp": torch.stack([_dla_cache[f"mlp_{l}"].cpu() for l in range(n_layer)]),
        "resid_dot": _dla_cache["resid_dot"].cpu(),
        "rms": _dla_cache["rms"].cpu(),
        "logit_diff": last_logits[yes_id] - last_logits[no_id],
        "top_id": int(last_logits.argmax()),
        "answer_text": processor.tokenizer.decode(last_logits.argmax().item()).strip(),
    }

In [9]:
results = {label: [] for label in samples}
rows = []

for label, entries in samples.items():
    for e in entries:
        q = question_by_id[e["question_id"]]
        yes_id, no_id = PAIRS[e["prefix"]]
        r = dla_question(resolve_image_path(q), q["question_text"], e["prefix"])
        total = r["embed"] + r["attn"].sum() + r["mlp"].sum()
        scale = r["embed"].abs() + r["attn"].abs().sum() + r["mlp"].abs().sum()
        cached_diff = e["answer_logits"][yes_id].float() - e["answer_logits"][no_id].float()
        if e["top_id"] == yes_id:
            sign_ok = bool(r["logit_diff"] > 0)
        elif e["top_id"] == no_id:
            sign_ok = bool(r["logit_diff"] < 0)
        else:
            sign_ok = True
        rows.append({
            "source": label,
            "rel_err_resid": ((total - r["resid_dot"]).abs() / scale).item(),
            "err_logit": (total / r["rms"] - r["logit_diff"]).abs().item(),
            "err_cache": (r["logit_diff"] - cached_diff).abs().item(),
            "answer_match": r["answer_text"] == e["answer_text"],
            "sign_ok": sign_ok,
            "in_pair": e["top_id"] in (yes_id, no_id),
            "finite": bool(torch.isfinite(total) and torch.isfinite(r["logit_diff"])),
        })
        results[label].append(r)

print(len(rows), "questions run")

/usr/local/lib/python3.12/dist-packages/transformers/models/qwen2_vl/image_processing_qwen2_vl.py:129: FutureWarning: Passing `min_pixels` and `max_pixels` to a processor call is deprecated and will be removed in v5.23. Pass in `size={'longest_edge': xxx, 'shortest_edge': xxx} to override the target size.`
  warnings.warn(


48 questions run


In [10]:
n = len(rows)
for label in samples:
    sub = [r for r in rows if r["source"] == label]
    print(f"{label:18s} rel err resid {max(r['rel_err_resid'] for r in sub):.5f} | "
          f"err logit {max(r['err_logit'] for r in sub):.4f} | err cache {max(r['err_cache'] for r in sub):.4f} | "
          f"answers match {sum(r['answer_match'] for r in sub)}/{len(sub)} | "
          f"sign ok {sum(r['sign_ok'] for r in sub)}/{len(sub)} | top-1 in pair {sum(r['in_pair'] for r in sub)}/{len(sub)}")

assert all(r["finite"] for r in rows), "non-finite values"
assert max(r["rel_err_resid"] for r in rows) < 0.05, "components do not sum to the final residual"
assert max(r["err_logit"] for r in rows) < 0.5, "components do not reproduce the logit difference"
assert max(r["err_cache"] for r in rows) < 0.25, "prompt or token ids differ from the question-aware cache"
assert sum(r["answer_match"] for r in rows) == n, "answers differ from the question-aware cache"
assert all(r["sign_ok"] for r in rows), "logit difference sign disagrees with the cached answer"
print("validation passed")

attribute/amber    rel err resid 0.00012 | err logit 0.0153 | err cache 0.0000 | answers match 16/16 | sign ok 16/16 | top-1 in pair 13/16
relation/amber     rel err resid 0.00013 | err logit 0.0114 | err cache 0.0000 | answers match 16/16 | sign ok 16/16 | top-1 in pair 16/16
relation/reefknot  rel err resid 0.00016 | err logit 0.0118 | err cache 0.0000 | answers match 16/16 | sign ok 16/16 | top-1 in pair 15/16
validation passed


In [11]:
for label, rs in results.items():
    attn = torch.stack([r["attn"] / r["rms"] for r in rs]).mean(0).flatten()
    mlp = torch.stack([r["mlp"] / r["rms"] for r in rs]).mean(0)
    mean_diff = torch.stack([r["logit_diff"] for r in rs]).mean().item()
    print(f"--- {label}: mean logit diff {mean_diff:.3f} ---")
    for i in attn.abs().topk(5).indices.tolist():
        print(f"  layer {i // n_head:2d} ({(i // n_head) / n_layer:.0%} depth) head {i % n_head:2d}  {attn[i]:+.3f}")
    for i in mlp.abs().topk(3).indices.tolist():
        print(f"  layer {i:2d} ({i / n_layer:.0%} depth) mlp      {mlp[i]:+.3f}")

--- attribute/amber: mean logit diff -5.603 ---
  layer 25 (89% depth) head 17  -0.179
  layer 21 (75% depth) head 10  +0.176
  layer 25 (89% depth) head 18  -0.140
  layer 21 (75% depth) head 23  -0.138
  layer 26 (93% depth) head 24  -0.131
  layer 23 (82% depth) mlp      -3.420
  layer 24 (86% depth) mlp      -1.574
  layer 22 (79% depth) mlp      -0.899
--- relation/amber: mean logit diff -5.968 ---
  layer 25 (89% depth) head 17  -0.205
  layer 21 (75% depth) head 10  +0.200
  layer 26 (93% depth) head 24  -0.126
  layer 26 (93% depth) head 22  +0.122
  layer 25 (89% depth) head 18  -0.121
  layer 23 (82% depth) mlp      -2.296
  layer 22 (79% depth) mlp      -1.505
  layer 24 (86% depth) mlp      -0.929
--- relation/reefknot: mean logit diff -0.959 ---
  layer 27 (96% depth) head 23  +0.385
  layer 27 (96% depth) head  5  +0.300
  layer 25 (89% depth) head 17  -0.196
  layer 23 (82% depth) head 13  +0.160
  layer 27 (96% depth) head 27  -0.126
  layer 23 (82% depth) mlp      -2.0